# Λόγια ή πράξεις; Πώς ταξιδεύει η βλάβη σε ένα δίκτυο LLM

Το σχέδιο και οι κανόνες ανάγνωσης είναι στο `docs/contagion_plan.md`,
γραμμένα πριν από τα runs.

- **Δίκτυο:** γραμμή 6 πρακτόρων, 0–1–2–3–4–5. Ο πράκτορας 0 είναι η «πηγή»,
  και ο πράκτορας d απέχει d βήματα από αυτήν.
- **Η πηγή** δεν καλεί μοντέλο: παίζει σταθερή κίνηση και στέλνει μηνύματα
  από τη λίστα του ίδιου μοντέλου στο corpus (`pools/`).

| κελί | λόγια της πηγής | πράξεις της πηγής |
|---|---|---|
| `source_cc` | συνεργατικά | πάντα συνεργασία (έλεγχος) |
| `source_hc` | εχθρικά | πάντα συνεργασία («φωνακλάς») |
| `source_cd` | συνεργατικά | πάντα αποστασία («ψεύτης») |
| `source_hd` | εχθρικά | πάντα αποστασία |
| `baseline` | ο 0 είναι κανονικός πράκτορας LLM | |

- **Ρυθμίσεις:** PD, 16 γύροι, 5 runs ανά κελί, διορθωμένο prompt δρομολόγησης.
- **Μοντέλα:** Llama-3.1-8B και Qwen2.5-7B.

## Δύο sessions, κάτω από 8 ώρες το καθένα

Το Llama μόνο του θέλει ~11 ώρες μίας κάρτας, άρα η καμπάνια δεν χωρά σε ένα
session. Χωρίζεται σε **μπλοκ runs**: το βήμα 1 τρέχει τα runs 1–3 **κάθε** κελιού
και των δύο μοντέλων, το βήμα 2 τα runs 4–5 (`--first-run 4`), με τους ίδιους
αριθμούς, σπόρους και ονόματα αρχείων που θα είχε ένα ενιαίο session. Κάθε κελί,
άρα και κάθε σύγκριση των κανόνων, παίρνει runs και από τα δύο sessions στην ίδια
αναλογία: μια διαφορά ανάμεσα στα sessions (π.χ. ενημέρωση βιβλιοθήκης στο Kaggle)
δεν μπορεί να περάσει για επίδραση κελιού.

| βήμα | runs | κάρτα 0 | κάρτα 1 | runs | ~ώρες |
|---|---|---|---|---|---|
| 1 | 1–3 | Llama: baseline, cc, hc · μετά Qwen: baseline | Llama: cd, hd · μετά Qwen: cc, hc, cd, hd | 30 | 5,5 |
| 2 | 4–5 | ίδια | ίδια | 20 | 3,7 |

Οι ώρες βγαίνουν από τα μετρημένα δευτερόλεπτα ανά κλήση (Llama ~9,4, Qwen2.5
~4,6, και η γραμμή ~15% πιο αργή όπως ο δακτύλιος), ±30%. Οι κανόνες διαβάζονται
μόνο όταν υπάρχουν και τα 5 runs: μετά το βήμα 2, τοπικά.

## Setup

1. Settings → Accelerator → **GPU T4 x2** (δύο κάρτες)
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach** (το Llama είναι gated)

Άλλαξε **μόνο** το `STEP` (1 ή 2) και μετά **Save Version → Save & Run All**. Στις
7,5 ώρες από την αρχή ό,τι τρέχει σταματά, και ό,τι τελείωσε πακετάρεται.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import time
T_START = time.time()     # το όριο των 7,5 ωρών μετρά από εδώ

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
assert N_GPU >= 2, ('Μία μόνο κάρτα: τα βήματα είναι υπολογισμένα για δύο κάρτες '
                    'παράλληλα -- Settings -> Accelerator -> GPU T4 x2')

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# Gate: χωρίς τον κώδικα της πηγής θα έτρεχαν απλές γραμμές, χωρίς --first-run
# τα δύο βήματα θα έγραφαν τα ίδια runs.
assert 'source_cc' in open('run_all_scenarios.py', encoding='utf-8').read(), \
    'Το repo δεν έχει τα κελιά της πηγής -- κάνε git push πρώτα.'
assert '--first-run' in open('campaign.py', encoding='utf-8').read(), \
    'Το campaign.py δεν έχει --first-run -- κάνε git push πρώτα.'
assert '[not read]' in open('contagion_report.py', encoding='utf-8').read(), \
    'Το contagion_report.py είναι παλιό -- κάνε git push πρώτα.'
LLA, Q25 = 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen2.5-7B-Instruct'
for m in (LLA, Q25):
    for kind in ('hostile', 'cooperative'):
        p = f"pools/{m.split('/')[-1]}_{kind}.json"
        assert os.path.exists(p), f'Λείπει η λίστα {p}'
assert 'Model loaded' in open('llm_client.py', encoding='utf-8').read(), \
    'το llm_client.py δεν τυπώνει «Model loaded» -- η κλειδαριά φόρτωσης δεν θα ξεκλείδωνε'
print('gate ok')

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    raise SystemExit(f'Λείπει το HF_TOKEN (χρειάζεται για το Llama): {e}')

## Ποιο βήμα

In [ ]:
STEP = 1          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ: 1 (runs 1-3) ή 2 (runs 4-5)

import glob, os

BLOCKS = {1: (1, 3), 2: (4, 2)}      # βήμα -> (πρώτο run, πόσα runs)
HOURS = {1: 5.5, 2: 3.7}
FIRST, N = BLOCKS[STEP]
# κάρτα -> [(μοντέλο, κελιά)]· και οι δύο κάρτες τρέχουν πρώτα το Llama, μετά το Qwen
LANES = ([(LLA, ['baseline', 'source_cc', 'source_hc']), (Q25, ['baseline'])],
         [(LLA, ['source_cd', 'source_hd']),
          (Q25, ['source_cc', 'source_hc', 'source_cd', 'source_hd'])])

def args_for(model, scenarios, out_root):
    return ['campaign.py', '--model', model, '--session', 'SRC', '--topology', 'line',
            '--n-agents', '6', '--topology-aware-comm-prompt', '--games', 'pd',
            '--conditions', 'cheap_talk', '--n-runs', str(N), '--first-run', str(FIRST),
            '--n-rounds', '16', '--source-pools', 'pools', '--scenarios', *scenarios,
            '--out-root', out_root, '--zip-mirror', '']

from campaign import plan
JOBS, EXPECTED, RESULT_DIR = [], {}, {}
for g, lane in enumerate(LANES):
    # Το Llama τρέχει και στις δύο κάρτες: δύο διεργασίες του ίδιου μοντέλου δεν
    # μοιράζονται φάκελο (το _progress.json και το zip του campaign είναι ανά φάκελο).
    out_root = f'/kaggle/working/lane{g}/results'
    jobs = []
    for model, scenarios in lane:
        name = f"{model.split('/')[-1]} · {', '.join(s.replace('source_', '') for s in scenarios)}"
        p = plan(model, 'SRC', 'line', N, 16, out_root, only=scenarios, games=['pd'],
                 comm_fix=True, conditions=['cheap_talk'], n_agents=6, source_pools='pools',
                 first_run=FIRST)
        EXPECTED[name], RESULT_DIR[name] = p['expected_runs'], p['result_dir']
        jobs.append((name, model, args_for(model, scenarios, out_root)))
    JOBS.append(jobs)

def progress(name):
    d = RESULT_DIR[name]
    return sum(1 for p in glob.glob(f'{d}/**/*.json', recursive=True)
               if not os.path.basename(p).startswith('_') and f'{os.sep}zips{os.sep}' not in p)

print(f'βήμα {STEP}: runs {FIRST}-{FIRST + N - 1} κάθε κελιού, ~{HOURS[STEP]} h')
for g, jobs in enumerate(JOBS):
    print(f'  κάρτα {g}: ' + '  ->  '.join(f'{n} [{EXPECTED[n]}]' for n, *_ in jobs))
print(f'  σύνολο {sum(EXPECTED.values())} runs')
assert sum(EXPECTED.values()) == 10 * N, 'λάθος πλήθος runs -- σταμάτα'

## Preview — κοστίζει μηδέν

Σε κάθε δουλειά: `topology : line`, τα σωστά runs, το σωστό `expecting`, φάκελο
`..._line_n6_commfix_cheaptalk` μέσα στον φάκελο της κάρτας, και στο βήμα 2
`--first-run 4` στην εντολή `RUN:`.

In [ ]:
import subprocess, sys

ok = True
for g, jobs in enumerate(JOBS):
    for name, model, argv in jobs:
        print('=' * 70)
        print(f'κάρτα {g} · {name}')
        r = subprocess.run([sys.executable, *argv, '--dry-run'], capture_output=True, text=True)
        out = r.stdout.strip()
        print(out[:900])
        if r.returncode != 0:
            ok = False
            print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
        runs = str(N) if FIRST == 1 else f'{FIRST}-{FIRST + N - 1}'
        for needle in ('topology     : line', f'(runs={runs} rounds=16',
                       f'expecting    : {EXPECTED[name]} run files', RESULT_DIR[name]):
            if needle not in out:
                ok = False
                print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')
        run_line = out.split('RUN:')[-1]
        if (FIRST != 1) != (f'--first-run {FIRST}' in run_line):
            ok = False
            print('ΛΑΘΟΣ ΑΡΙΘΜΗΣΗ RUNS στην εντολή')
        if '--zip-mirror' in run_line:
            ok = False
            print('ΤΟ MIRROR ΕΙΝΑΙ ΕΝΕΡΓΟ')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση: δύο κάρτες, μία ουρά η καθεμία

Κάθε δουλειά είναι ένα `campaign.py` με δικό του log στο `/kaggle/working/logs/`.
Εδώ τυπώνεται η γραμμή `device map` κάθε μοντέλου μόλις φορτώσει, η πρόοδος κάθε
10 λεπτά, κάθε δουλειά που τελειώνει, και ο απολογισμός.

In [ ]:
import collections, glob, os, shutil, signal, subprocess, sys, threading, time

LOG_DIR = '/kaggle/working/logs'
POLL_S, REPORT_S = 30, 600
# 7,5 ώρες από την αρχή του notebook: ό,τι τρέχει τότε σταματά, και η
# συσκευασία παρακάτω προλαβαίνει πριν από το όριο των 8 ωρών.
DEADLINE = T_START + 7.5 * 3600
os.makedirs(LOG_DIR, exist_ok=True)

try:
    from huggingface_hub import constants as _hfc
    HF_CACHE = _hfc.HF_HUB_CACHE
except Exception:
    HF_CACHE = os.path.expanduser('~/.cache/huggingface/hub')

def cache_dir(model):
    return os.path.join(HF_CACHE, 'models--' + model.replace('/', '--'))

def log_has(path, needle):
    try:
        with open(path, encoding='utf-8', errors='replace') as fh:
            return needle in fh.read()
    except FileNotFoundError:
        return False

def stop(proc):
    # Κάθε δουλειά έχει δική της ομάδα διεργασιών: το campaign.py τρέχει το
    # run_all_scenarios.py ως παιδί, και ένα σκέτο terminate θα το άφηνε να
    # συνεχίζει μόνο του στην κάρτα.
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except Exception:
        proc.terminate()

# Τα αρχεία ενός μοντέλου σβήνονται από τον δίσκο μόλις το έχουν φορτώσει όλες
# οι δουλειές του, και στις δύο κάρτες: μετά τη φόρτωση δεν τα διαβάζει τίποτα.
USES = collections.Counter(model for jobs in JOBS for _, model, _ in jobs)
loaded = collections.Counter()
load_lock = threading.Lock()   # ένα μοντέλο φορτώνει τη φορά
guard = threading.Lock()       # προστατεύει state, finished, procs, loaded
state, finished, procs = {}, [], {}

def release(model):
    with guard:
        loaded[model] += 1
        last = loaded[model] == USES[model]
    if last:
        shutil.rmtree(cache_dir(model), ignore_errors=True)

def run_lane(gpu, jobs):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2')
    for i, (name, model, argv) in enumerate(jobs):
        log = os.path.join(LOG_DIR, f"step{STEP}_gpu{gpu}_{i}_{model.split('/')[-1]}.log")
        t = time.time()
        proc, released, status = None, False, 'SKIPPED (χρόνος)'
        try:
            with load_lock:
                # Το όριο ελέγχεται αφού πάρει την κλειδαριά: η αναμονή μπορεί να
                # το περάσει, και καμία δουλειά δεν ξεκινά μετά από αυτό.
                if time.time() < DEADLINE:
                    with guard:
                        state[gpu] = dict(name=name, log=log, t0=t, phase='φόρτωση')
                    with open(log, 'w', encoding='utf-8') as fh:
                        proc = subprocess.Popen([sys.executable, *argv], stdout=fh,
                                                stderr=subprocess.STDOUT, env=env,
                                                start_new_session=True)
                    with guard:
                        procs[gpu] = proc
                    while (proc.poll() is None and time.time() < DEADLINE
                           and not log_has(log, '[local] Model loaded')):
                        time.sleep(10)
            release(model)
            released = True
            if proc is not None:
                with guard:
                    state[gpu]['phase'] = 'τρέχει'
                rc = proc.wait()
                status = 'OK' if rc == 0 else f'FAILED (exit {rc})'
        except Exception as e:   # ποτέ σιωπηλά: η δουλειά μπαίνει στον απολογισμό
            status = f'ERROR {e!r}'[:60]
        if not released:
            release(model)
        with guard:
            finished.append((gpu, name, status, (time.time() - t) / 60, log))
    with guard:
        state[gpu] = None

threads = []
for gpu, jobs in enumerate(JOBS):
    th = threading.Thread(target=run_lane, args=(gpu, jobs), daemon=True)
    th.start()
    threads.append(th)
    time.sleep(2)   # η κάρτα 0 παίρνει πρώτη την κλειδαριά φόρτωσης

shown, n_reported, last, stopped = set(), 0, time.time(), False
while any(th.is_alive() for th in threads):
    time.sleep(POLL_S)
    if time.time() > DEADLINE:
        if not stopped:
            stopped = True
            print('7,5 ώρες: σταματώ ό,τι τρέχει. Ό,τι τελείωσε πακετάρεται παρακάτω.', flush=True)
        with guard:
            for p in procs.values():
                if p.poll() is None:
                    stop(p)
    with guard:
        snap = {g: dict(s) for g, s in state.items() if s}
        done = list(finished)
    for g, s in snap.items():
        if s['log'] in shown:
            continue
        try:
            with open(s['log'], encoding='utf-8', errors='replace') as fh:
                line = next((l for l in fh if '[local] device map' in l), None)
        except FileNotFoundError:
            line = None
        if line:
            shown.add(s['log'])
            print(f"κάρτα {g} · {s['name']}: {line.strip()}", flush=True)
    for g, name, st, mins, _ in done[n_reported:]:
        print(f'--> κάρτα {g} · {name}: {st}, {mins:.0f} λεπτά', flush=True)
    n_reported = len(done)
    if time.time() - last >= REPORT_S:
        last = time.time()
        print(f'--- {(last - T_START) / 3600:.1f} h από την αρχή ---', flush=True)
        for g, s in sorted(snap.items()):
            print(f"  κάρτα {g}: {s['name']:38s} {s['phase']:8s} "
                  f"{progress(s['name']):4d}/{EXPECTED[s['name']]}, "
                  f"{(time.time() - s['t0']) / 3600:.1f} h", flush=True)

for th in threads:
    th.join()
for g, name, st, mins, _ in finished[n_reported:]:
    print(f'--> κάρτα {g} · {name}: {st}, {mins:.0f} λεπτά')

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for g, name, st, mins, _ in finished:
    print(f'  κάρτα {g}  {st:24s} {mins:6.0f} λ   {name}   [{progress(name)}/{EXPECTED[name]}]')
print(f'\nσύνολο {(time.time() - T_START) / 3600:.1f} h από την αρχή του notebook '
      f'(εκτίμηση ~{HOURS[STEP]} h)')
for g, name, st, mins, log in finished:
    if st != 'OK' and os.path.exists(log):
        print('\n' + '-' * 70)
        print(f'ΑΠΟΤΥΧΙΑ: κάρτα {g} · {name} -- οι τελευταίες γραμμές του {log}:')
        with open(log, encoding='utf-8', errors='replace') as fh:
            print(''.join(fh.readlines()[-25:]))

## Μάζεμα, και πρώτη ματιά

Τα runs και των δύο καρτών μπαίνουν σε ένα δέντρο ανά μοντέλο, όπως θα μπουν
τοπικά. Το `contagion_report.py` δείχνει τη συνεργασία ανά απόσταση· τους κανόνες
R1–R3 δεν τους διαβάζει πριν υπάρχουν και τα 5 runs κάθε κελιού.

In [ ]:
BOX_NAME = f'contagion_step{STEP}'
import collections, glob, json, os, shutil

BOX = f'/kaggle/working/{BOX_NAME}'
shutil.rmtree(BOX, ignore_errors=True)
copied, broken = collections.Counter(), []
for name, d in RESULT_DIR.items():
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        try:
            with open(p, encoding='utf-8') as fh:
                json.load(fh)
        except Exception:
            broken.append(p)    # run που κόπηκε την ώρα που γραφόταν
            continue
        dst = os.path.join(BOX, os.path.basename(d), os.path.relpath(p, d))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)
        copied[name] += 1
for name in RESULT_DIR:
    flag = '' if copied[name] == EXPECTED[name] else '   <- λείπουν'
    print(f'  {name:38s} {copied[name]:3d}/{EXPECTED[name]} runs{flag}')
if broken:
    print(f'  {len(broken)} μισογραμμένα αρχεία έμειναν έξω')
shutil.copytree(LOG_DIR, os.path.join(BOX, f'_logs_step{STEP}'), dirs_exist_ok=True)
TREES = sorted(p for p in glob.glob(f'{BOX}/*') if not os.path.basename(p).startswith('_'))

import subprocess, sys
r = subprocess.run([sys.executable, 'contagion_report.py', '--roots', *TREES],
                   capture_output=True, text=True)
print(r.stdout[-8000:]); print(r.stderr[-1500:])
out = shutil.make_archive(BOX, 'zip', BOX)
print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')

## Μετά

Κατέβασε το `contagion_step<N>.zip` και αποσυμπίεσέ το στο
`diplomatikh/contagion_runs/`. Τα δύο βήματα ενώνονται στους ίδιους φακέλους (τα
runs είναι αριθμημένα 01–05 και δεν συγκρούονται). Τα runs **δεν** μπαίνουν στο
tracker: είναι παραλλαγές (σενάριο πηγής, έξι πράκτορες), όπως τα `harm_source/`,
και το ingest σωστά τα αρνείται. Όταν υπάρχουν και τα δύο βήματα, η ανάλυση (από
τον φάκελο `diplomatikh`):

```
python cheaptalk_bench/contagion_report.py --roots contagion_runs/*_line_n6_commfix_cheaptalk
```